# PHASE 2: Data Cleaning for further EDA Analysis

In [1]:
# Helper functions
import pandas as pd
import numpy as np
from pathlib import Path

def missing_info(df):
    missing_info = pd.DataFrame({
        'Missing Count': df.isna().sum(),
        'Percent': (df.isna().mean() * 100).round(2),
        'Data type': df.dtypes,
    })
    
    missing_info = missing_info[missing_info['Missing Count'] > 0]
    return missing_info.sort_values('Percent',ascending=False)

def is_duplicates(df):
    print(f'Total Duplicates: {df.duplicated().sum()}')

def dedup(df: pd.DataFrame) -> pd.DataFrame: # should be done 1st
    """
    Remove duplicate property-room records based on (id, occupancy).
    The first occurrence is retained.

        - Duplicate (id, occupancy) pairs
    """

    df = df.drop_duplicates(subset=['id', 'occupancy'], keep='first')
    return df

def drop_columns(df: pd.DataFrame, columns: list[str] | None = None) -> pd.DataFrame:
    """
    Drop columns that are not useful for modeling.

    If columns is not provided, the default set of columns
    identified during EDA will be removed.

    col = ['id', 'title', 'address', 'gate_closing_time', 'total_bathroom']
    """

    drop_cols = ['id', 'title', 'address', 'total_bathrooms', 'warden', 'cooking_allowed', 'gate_closing_time', 'guardian_required', 'nonveg_allowed', 'smoking_allowed']
    if columns is None:
        columns = drop_cols

    df = df.drop(columns=columns)
    return df

def drop_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop rows that are not useful for modelling.
    
    this func() drops the reows which,
    - ~1% rows with missing values in key columns
    - rent with 0 or NaNs
    - occupancy is NaN
    """

    df = df.dropna(subset=['rent', 'deposit', 'occupancy', 'attached_bathroom'])
    # Remove invalid/placeholder rents.
    # The minimum realistic PG rent in Chennai is well above 1000.
    df = df[df['rent'] >= 1000]

    return df

def fill_boolean_amenities(df: pd.DataFrame) -> pd.DataFrame:
    """
    FIll missing boolean amenites with False
    then convert the columns to bool type
    """
    bool_cols = ['attached_bathroom', 'mess', 'wifi', 'laundry', 'power_backup',
        'refrigerator', 'common_tv', 'room_cleaning','room_ac', 
        'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding',
        'room_attached_bath',
    ]   

    # imputation for amenities
    df[bool_cols] = df[bool_cols].fillna(False).astype(bool)

    assert df[bool_cols].isna().sum().sum() == 0 # should be 0
    assert (df[bool_cols].dtypes == bool).all() # should show: bool

    return df

def impute_transit_score(df: pd.DataFrame) -> pd.DataFrame:
    # this is for fix the left influend skew-ness (should be done before imputation)
    df['transit_score'] = df['transit_score'].replace(-10, np.nan)

    # creating tag for msiing values rows
    df['transit_score_missing'] = df['transit_score'].isna().astype(int)

    # impute with local (locality) median
    df['transit_score'] = (
         
         df.groupby('locality')['transit_score']
        .transform(lambda x: x.fillna(x.median()))
    )

    # impute with global median (if local median is Nan)
    df['transit_score'] = df['transit_score'].fillna(df['transit_score'].median())

    return df

def impute_lifestyle_score(df: pd.DataFrame) -> pd.DataFrame:
    
   # imputation for lifestyle_score
    df['lifestyle_score_missing'] = df['lifestyle_score'].isna().astype(int)  # creating tag for msiing values rows

    df['lifestyle_score'] = ( 
            df.groupby('locality')['lifestyle_score']
            .transform(lambda x: x.fillna(x.median()))
    )  # impute with local (locality) median

    # impute with global median (if local median is Nan)
    df['lifestyle_score'] = df['lifestyle_score'].fillna(df['lifestyle_score'].median()) # impute with global median (if local median is Nan)

    return df

def save_csv(df: pd.DataFrame):
    output_dir = Path('../Data/processed/EDA')
    output_dir.mkdir(parents=True, exist_ok=True)

    # save cleaned dataset
    df.to_csv(output_dir / '01_eda_Phase-2_processed.csv', index=False)
    print("Dataset saved successfully!")

## 2.1 Drop Rows & Columns

In [2]:
df = pd.read_csv(r'D:\Hustle\Chennai-PG\Data\raw\chennai_pg_dataset.csv') 

In [4]:
missing_info(df)

,Missing Count,Percent,Data type
gate_closing_time,1476,82.78,str
lifestyle_score,900,50.48,float64
transit_score,900,50.48,float64
mess,834,46.78,object
cooking_allowed,833,46.72,object
wifi,830,46.55,object
power_backup,830,46.55,object
common_tv,829,46.49,object
refrigerator,829,46.49,object
room_cleaning,806,45.20,object


In [3]:
is_duplicates(df)

Total Duplicates: 162


In [4]:
df = dedup(df)
df = drop_columns(df)
df = drop_rows(df)
df = fill_boolean_amenities(df)
df = impute_transit_score(df)
df = impute_lifestyle_score(df)
# save_csv(df)

In [5]:
is_duplicates(df)

Total Duplicates: 0


In [6]:
missing_info(df)

,Missing Count,Percent,Data type
parking,144,9.79,str


In [7]:
df.shape

(1471, 31)

# EDA Phase 2: Data Cleaning and Dataset Preparation Report

> **Purpose:** Execute every cleaning decision documented in Phase 1 and produce a clean dataset for all remaining EDA phases.
> No feature analysis or modelling transformations are applied here.
> The raw dataset is preserved untouched throughout.

---

## Dataset State Entering Phase 2

- Source: raw scraped NoBroker dataset
- Issues identified in Phase 1: duplicates, missing values, incorrect dtypes, sentinel values, low signal columns

---

## Cleaning Steps Performed

### 1. Duplicate Record Removal

#### Observations

- The scraper collected listings by looping through an AREAS dictionary that contained overlapping locality definitions
- The same physical PG appeared multiple times in the raw data when a locality was indexed under two different area names
- Not all repeated property IDs are true duplicates — many PGs offer multiple room configurations (Single, Double, Three Sharing, Four Sharing), each with a distinct rent
- Deduplicating on property ID alone would incorrectly remove valid room tier variants

#### Decision

- Deduplication key: composite of `id` and `occupancy`
- Only rows where both values are identical are treated as duplicates
- First occurrence of each unique pair is kept

```python
df_clean = df.drop_duplicates(subset=['id', 'occupancy'], keep='first')
```

---

### 2. Removing Unnecessary Features

#### Observations

- Ten columns carry no predictive signal or are structurally redundant

#### Decision

| Column | Reason for Dropping |
|---|---|
| `id` | Unique identifier, zero predictive value |
| `title` | Free text listing name, redundant with locality and gender |
| `address` | Full address string, redundant with locality and coordinates |
| `total_bathrooms` | Building level aggregate, median is 0, redundant with `room_attached_bath` |
| `gate_closing_time` | Over 80% missing, inconsistent format, low predictive signal |
| `warden` | Low variance, limited influence on rent |
| `cooking_allowed` | Low variance, limited influence on rent |
| `guardian_required` | Low variance, limited influence on rent |
| `nonveg_allowed` | Low variance, limited influence on rent |
| `smoking_allowed` | Low variance, limited influence on rent |

---

### 3. Removing Invalid Rows

#### Observations

- Four columns are core to the task: `rent`, `deposit`, `occupancy`, `attached_bathroom`
- Rows missing any of these cannot contribute meaningful observations
- Rent values below Rs. 1,000 are not realistic for any Chennai PG accommodation
- Values below this threshold are scraping artifacts or data entry errors
- 24 rows had NaN in the `occupancy` column — too few to impute meaningfully and incompatible with the dedup key

#### Decision

| Condition | Action | Rationale |
|---|---|---|
| `rent` is NaN or below Rs. 1,000 | Drop rows | Target variable must be valid and realistic |
| `deposit` is NaN | Drop rows | Core feature, imputing deposit is unsound |
| `occupancy` is NaN (24 rows) | Drop rows | Breaks the dedup composite key |
| `attached_bathroom` is NaN | Drop rows | Core feature for the task |

---

### 4. Boolean Amenity Feature Cleaning

#### Observations

- Fourteen amenity columns were stored as `object` dtype in the raw data
- Their only valid values are Python `True` and `False`
- Missing values in these columns mean the PG owner did not disclose the amenity — validated against live NoBroker JSON API responses
- The semantics of NaN here is "not disclosed", treated as unavailable under the conservative modeling assumption
- Fill must happen before dtype conversion — filling after conversion can silently produce incorrect True values

#### Decision

- Step 1: Fill all NaN values with `False`
- Step 2: Cast to `bool` dtype

```python
df_clean[amenity_cols] = df_clean[amenity_cols].fillna(False).astype(bool)
```

#### Columns Cleaned

`attached_bathroom`, `mess`, `wifi`, `laundry`, `power_backup`, `refrigerator`, `common_tv`, `room_cleaning`, `room_ac`, `room_cupboard`, `room_tv`, `room_geyser`, `room_bedding`, `room_attached_bath`

---

### 5. Transit Score Cleaning

#### Observations

- Column contained a sentinel value of negative 10
- This value was used by the NoBroker API to signal that no transit data was available — it is not a real score
- Retaining negative 10 as a numeric value would introduce severe left skew and corrupt any analysis or model that uses this feature
- Approximately 50% of rows had no transit score — these correspond to localities that NoBroker has not indexed for transit connectivity

#### Decision

| Step | Action | Rationale |
|---|---|---|
| 1 | Replace negative 10 with NaN | Correctly represent missingness |
| 2 | Create `transit_score_missing` binary indicator | Preserve missingness signal for the model |
| 3 | Impute NaN using median within each locality | Properties in the same locality share similar transit characteristics |
| 4 | Fallback: fill remaining NaN with global median | Handles entire localities with no valid scores |

---

### 6. Lifestyle Score Cleaning

#### Observations

- Missing for the same approximately 900 rows as `transit_score` — confirmed to be the identical set of rows
- NoBroker does not score every locality for lifestyle amenities — the missingness is structural, not random

#### Decision

| Step | Action |
|---|---|
| 1 | Create `lifestyle_score_missing` binary indicator |
| 2 | Impute NaN using median within each locality |
| 3 | Fallback: fill remaining NaN with global median |

---

### 7. New Indicator Features Added

| Column | Value | Meaning |
|---|---|---|
| `transit_score_missing` | 1 | Transit score was originally absent for this row |
| `transit_score_missing` | 0 | Transit score was present and valid |
| `lifestyle_score_missing` | 1 | Lifestyle score was originally absent for this row |
| `lifestyle_score_missing` | 0 | Lifestyle score was present and valid |

- Both indicators show the same True count, confirming both scores were missing for the exact same set of rows
- These features allow the model to learn whether being in an unscored locality is itself predictive of rent

---

## Validation Checks

| Check | Expected Result | Status |
|---|---|---|
| Duplicate `(id, occupancy)` pairs remaining | 0 | Passed |
| NaN in any boolean amenity column | 0 | Passed |
| Dtype of all boolean amenity columns | `bool` | Passed |
| Sentinel value of negative 10 in `transit_score` | None remaining | Passed |
| NaN in `transit_score` | 0 | Passed |
| NaN in `lifestyle_score` | 0 | Passed |
| Rent values below Rs. 1,000 or NaN | None remaining | Passed |

---

## Output Dataset

- Saved to: `Data/processed/EDA/01_eda_Phase-2_processed.csv`
- Shape entering Phase 3: 1,601 rows × 35 columns
- Raw dataset preserved separately and will not be modified

---

## What Has Been Deferred

- All of the following belong to `03_preprocessing.ipynb` and are deliberately held back until EDA is complete:

| Transformation | Reason for Deferral |
|---|---|
| One hot encoding for nominal categorical features | Applying before EDA contaminates distribution analysis |
| Smoothed target encoding for locality | Requires train/test split to avoid data leakage |
| Log transform of rent and deposit | Deferred until EDA confirms skew and transformation need |
| Feature scaling and normalization | Model stage decision, not EDA |
| Train / validation / test splitting | Must happen after all EDA decisions are finalised |

---

## Remaining EDA Plan

### PHASE 3: Univariate Analysis (One feature at a time)

- Target variable rent studied in isolation first: distribution, skew, outliers, log transform assessment
- Each numeric feature (deposit, transit score, lifestyle score) analysed for distribution shape and outliers
- Each categorical feature (locality, gender, available for, occupancy, parking) analysed for cardinality and frequency
- Each boolean feature analysed for prevalence and near zero variance

### PHASE 4: RELATIONSHIP ANALYSIS (FEATURE VS TARGET)

- Bivariate analysis: every feature plotted or compared against rent
- Correlation matrix across all numeric features to detect multicollinearity
- Diagnostic tree model fitted purely to rank feature importance
- Final decisions on which features to drop before preprocessing

---

> **Next:** `03_EDA_Phase3.ipynb` — univariate feature analysis on the cleaned dataset produced here.